In [8]:
import re

# Unique tokens for algorithm status
NIL = []
FAILURE = None

class Term:
    pass

class Var(Term):
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return str(self.name)

class Const(Term):
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return str(self.name)

class Predicate(Term):
    def __init__(self, symbol, args):
        self.symbol = symbol
        self.args = list(args)
    def __repr__(self):
        return f"{self.symbol}({', '.join(map(str, self.args))})"


# ==========================================
# RECURSIVE DESCENT PARSER FOR FOL TEXT
# ==========================================
def parse_expression(text):
    """
    Parses a string like 'p(b, X, f(g(Z)))' into structural Term objects.
    Assumes variables are uppercase letters (X, Y, Z) and constants are lowercase.
    """
    # Clean whitespace and tokenize words, commas, and parentheses
    tokens = re.findall(r'[a-zA-Z0-9]+|[\(\),]', text.strip())

    def parse_token_list(token_list):
        if not token_list:
            raise ValueError("Unexpected end of input")

        token = token_list.pop(0)

        # Check if it's a function or predicate: name followed by '('
        if token_list and token_list[0] == '(':
            symbol = token
            token_list.pop(0)  # consume '('

            args = []
            while token_list and token_list[0] != ')':
                args.append(parse_token_list(token_list))
                if token_list and token_list[0] == ',':
                    token_list.pop(0)  # consume ','

            if token_list and token_list[0] == ')':
                token_list.pop(0)  # consume ')'
            else:
                raise ValueError(f"Missing closing parenthesis for '{symbol}'")

            return Predicate(symbol, args)

        # If it's a single word token
        else:
            # Standard logic heuristic: Uppercase = Variable, Lowercase = Constant
            if token[0].isupper():
                return Var(token)
            else:
                return Const(token)

    try:
        return parse_token_list(tokens)
    except Exception as e:
        raise ValueError(f"Parsing error on input structural layout: {e}")


# ==========================================
# UNIFICATION LOGIC WITH TRACE
# ==========================================
def occurs_in(var, expr):
    if isinstance(expr, Var):
        return var.name == expr.name
    if isinstance(expr, Predicate):
        return any(occurs_in(var, arg) for arg in expr.args)
    return False

def apply_substitution(subst_list, expr):
    if not subst_list:
        return expr
    current_expr = expr
    for s_var, s_val in subst_list:
        current_expr = _substitute_single(s_var, s_val, current_expr)
    return current_expr

def _substitute_single(var_name, val_term, expr):
    if isinstance(expr, Var):
        return val_term if expr.name == var_name else expr
    if isinstance(expr, Const):
        return expr
    if isinstance(expr, Predicate):
        return Predicate(expr.symbol, [_substitute_single(var_name, val_term, arg) for arg in expr.args])
    return expr

step_counter = 0

def unify(psi1, psi2):
    global step_counter

    # Step 1: Variables / Constants
    if isinstance(psi1, (Var, Const)) or isinstance(psi2, (Var, Const)):
        if type(psi1) == type(psi2) and psi1.name == psi2.name:
            return NIL
        elif isinstance(psi1, Var):
            if occurs_in(psi1, psi2): return FAILURE
            return [(psi1.name, psi2)]
        elif isinstance(psi2, Var):
            if occurs_in(psi2, psi1): return FAILURE
            return [(psi2.name, psi1)]
        else:
            return FAILURE

    # Step 2 & 3: Signature validation
    if psi1.symbol != psi2.symbol or len(psi1.args) != len(psi2.args):
        return FAILURE

    # Step 4 & 5: Iterative substitution processing
    SUBST = NIL
    L1_args = list(psi1.args)
    L2_args = list(psi2.args)

    for i in range(len(L1_args)):
        S = unify(L1_args[i], L2_args[i])

        if S == FAILURE:
            return FAILURE

        if S != NIL:
            step_counter += 1
            for var_name, val in S:
                print(f"SUBST θ= {{{var_name}/{val}}}")

            for j in range(i, len(L1_args)):
                L1_args[j] = apply_substitution(S, L1_args[j])
                L2_args[j] = apply_substitution(S, L2_args[j])

            current_psi1 = Predicate(psi1.symbol, L1_args)
            current_psi2 = Predicate(psi2.symbol, L2_args)
            print(f"S_{step_counter} => {{ {current_psi1}; {current_psi2} }}\n")

            SUBST = S + SUBST

    return SUBST


# ==========================================
# MAIN INTERACTION LOOP
# ==========================================
if __name__ == "__main__":
    print("--- First-Order Logic Unification Engine ---")
    print("Rule: Uppercase letters (X, Y, Z) are parsed as Variables.")
    print("      Lowercase tokens (b, john, a) are parsed as Constants.\n")

    # Quick defaults for rapid testing if user presses enter
    default1 = "p(b, X, f(g(Z)))"
    default2 = "p(Z, f(Y), f(Y))"

    input1 = input(f"Enter Expression 1 [Leave blank for '{default1}']: ").strip()
    input2 = input(f"Enter Expression 2 [Leave blank for '{default2}']: ").strip()

    if not input1: input1 = default1
    if not input2: input2 = default2

    try:
        expr1 = parse_expression(input1)
        expr2 = parse_expression(input2)

        print("\n--- Starting Trace Execution ---")
        print(f"Expression1 = {expr1}")
        print(f"Expression2 = {expr2}")
        print(f"S_0 => {{ {expr1}; {expr2} }}\n")

        step_counter = 0
        final_subst = unify(expr1, expr2)

        if final_subst is not FAILURE:
            print("Unified Successfully.")
            print(f"Final Complete Substitution Map (MGU): {final_subst if final_subst else 'NIL'}")
        else:
            print("Unification Failed.")

    except ValueError as err:
        print(f"\n[Syntax Error] {err}")

--- First-Order Logic Unification Engine ---
Rule: Uppercase letters (X, Y, Z) are parsed as Variables.
      Lowercase tokens (b, john, a) are parsed as Constants.

Enter Expression 1 [Leave blank for 'p(b, X, f(g(Z)))']: 
Enter Expression 2 [Leave blank for 'p(Z, f(Y), f(Y))']: 

--- Starting Trace Execution ---
Expression1 = p(b, X, f(g(Z)))
Expression2 = p(Z, f(Y), f(Y))
S_0 => { p(b, X, f(g(Z))); p(Z, f(Y), f(Y)) }

SUBST θ= {Z/b}
S_1 => { p(b, X, f(g(b))); p(b, f(Y), f(Y)) }

SUBST θ= {X/f(Y)}
S_2 => { p(b, f(Y), f(g(b))); p(b, f(Y), f(Y)) }

SUBST θ= {Y/g(b)}
S_3 => { f(g(b)); f(g(b)) }

SUBST θ= {Y/g(b)}
S_4 => { p(b, f(Y), f(g(b))); p(b, f(Y), f(g(b))) }

Unified Successfully.
Final Complete Substitution Map (MGU): [('Y', g(b)), ('X', f(Y)), ('Z', b)]


In [9]:
import re

# Unique tokens for algorithm status
NIL = []
FAILURE = None

class Term:
    pass

class Var(Term):
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return str(self.name)

class Const(Term):
    def __init__(self, name):
        self.name = name
    def __repr__(self):
        return str(self.name)

class Predicate(Term):
    def __init__(self, symbol, args):
        self.symbol = symbol
        self.args = list(args)
    def __repr__(self):
        return f"{self.symbol}({', '.join(map(str, self.args))})"


# ==========================================
# RECURSIVE DESCENT PARSER FOR FOL TEXT
# ==========================================
def parse_expression(text):
    """
    Parses a string like 'p(b, X, f(g(Z)))' into structural Term objects.
    Assumes variables are uppercase letters (X, Y, Z) and constants are lowercase.
    """
    # Clean whitespace and tokenize words, commas, and parentheses
    tokens = re.findall(r'[a-zA-Z0-9]+|[\(\),]', text.strip())

    def parse_token_list(token_list):
        if not token_list:
            raise ValueError("Unexpected end of input")

        token = token_list.pop(0)

        # Check if it's a function or predicate: name followed by '('
        if token_list and token_list[0] == '(':
            symbol = token
            token_list.pop(0)  # consume '('

            args = []
            while token_list and token_list[0] != ')':
                args.append(parse_token_list(token_list))
                if token_list and token_list[0] == ',':
                    token_list.pop(0)  # consume ','

            if token_list and token_list[0] == ')':
                token_list.pop(0)  # consume ')'
            else:
                raise ValueError(f"Missing closing parenthesis for '{symbol}'")

            return Predicate(symbol, args)

        # If it's a single word token
        else:
            # Standard logic heuristic: Uppercase = Variable, Lowercase = Constant
            if token[0].isupper():
                return Var(token)
            else:
                return Const(token)

    try:
        return parse_token_list(tokens)
    except Exception as e:
        raise ValueError(f"Parsing error on input structural layout: {e}")


# ==========================================
# UNIFICATION LOGIC WITH TRACE
# ==========================================
def occurs_in(var, expr):
    if isinstance(expr, Var):
        return var.name == expr.name
    if isinstance(expr, Predicate):
        return any(occurs_in(var, arg) for arg in expr.args)
    return False

def apply_substitution(subst_list, expr):
    if not subst_list:
        return expr
    current_expr = expr
    for s_var, s_val in subst_list:
        current_expr = _substitute_single(s_var, s_val, current_expr)
    return current_expr

def _substitute_single(var_name, val_term, expr):
    if isinstance(expr, Var):
        return val_term if expr.name == var_name else expr
    if isinstance(expr, Const):
        return expr
    if isinstance(expr, Predicate):
        return Predicate(expr.symbol, [_substitute_single(var_name, val_term, arg) for arg in expr.args])
    return expr

step_counter = 0

def unify(psi1, psi2):
    global step_counter

    # Step 1: Variables / Constants
    if isinstance(psi1, (Var, Const)) or isinstance(psi2, (Var, Const)):
        if type(psi1) == type(psi2) and psi1.name == psi2.name:
            return NIL
        elif isinstance(psi1, Var):
            if occurs_in(psi1, psi2): return FAILURE
            return [(psi1.name, psi2)]
        elif isinstance(psi2, Var):
            if occurs_in(psi2, psi1): return FAILURE
            return [(psi2.name, psi1)]
        else:
            return FAILURE

    # Step 2 & 3: Signature validation
    if psi1.symbol != psi2.symbol or len(psi1.args) != len(psi2.args):
        return FAILURE

    # Step 4 & 5: Iterative substitution processing
    SUBST = NIL
    L1_args = list(psi1.args)
    L2_args = list(psi2.args)

    for i in range(len(L1_args)):
        S = unify(L1_args[i], L2_args[i])

        if S == FAILURE:
            return FAILURE

        if S != NIL:
            step_counter += 1
            for var_name, val in S:
                print(f"SUBST θ= {{{var_name}/{val}}}")

            for j in range(i, len(L1_args)):
                L1_args[j] = apply_substitution(S, L1_args[j])
                L2_args[j] = apply_substitution(S, L2_args[j])

            current_psi1 = Predicate(psi1.symbol, L1_args)
            current_psi2 = Predicate(psi2.symbol, L2_args)
            print(f"S_{step_counter} => {{ {current_psi1}; {current_psi2} }}\n")

            SUBST = S + SUBST

    return SUBST


# ==========================================
# MAIN INTERACTION LOOP
# ==========================================
if __name__ == "__main__":
    print("--- First-Order Logic Unification Engine ---")
    print("Rule: Uppercase letters (X, Y, Z) are parsed as Variables.")
    print("      Lowercase tokens (b, john, a) are parsed as Constants.\n")

    # Quick defaults for rapid testing if user presses enter
    default1 = "p(b, X, f(g(Z)))"
    default2 = "p(Z, f(Y), f(Y))"

    input1 = input(f"Enter Expression 1 [Leave blank for '{default1}']: ").strip()
    input2 = input(f"Enter Expression 2 [Leave blank for '{default2}']: ").strip()

    if not input1: input1 = default1
    if not input2: input2 = default2

    try:
        expr1 = parse_expression(input1)
        expr2 = parse_expression(input2)

        print("\n--- Starting Trace Execution ---")
        print(f"Expression1 = {expr1}")
        print(f"Expression2 = {expr2}")
        print(f"S_0 => {{ {expr1}; {expr2} }}\n")

        step_counter = 0
        final_subst = unify(expr1, expr2)

        if final_subst is not FAILURE:
            print("Unified Successfully.")
            print(f"Final Complete Substitution Map (MGU): {final_subst if final_subst else 'NIL'}")
        else:
            print("Unification Failed.")

    except ValueError as err:
        print(f"\n[Syntax Error] {err}")

--- First-Order Logic Unification Engine ---
Rule: Uppercase letters (X, Y, Z) are parsed as Variables.
      Lowercase tokens (b, john, a) are parsed as Constants.

Enter Expression 1 [Leave blank for 'p(b, X, f(g(Z)))']: Q(y,G(A,B))
Enter Expression 2 [Leave blank for 'p(Z, f(Y), f(Y))']: Q(G(x,x),y)

--- Starting Trace Execution ---
Expression1 = Q(y, G(A, B))
Expression2 = Q(G(x, x), y)
S_0 => { Q(y, G(A, B)); Q(G(x, x), y) }

Unification Failed.
